# Transfer Learning ResNet18 untuk Klasifikasi Citra Sampah

**Praktikum Computer Vision & Deep Learning — Google Colab → GitHub**

Notebook ini disusun mengikuti alur repo contoh RET503: unggah dataset ZIP, cek distribusi kelas, lakukan training, evaluasi, simpan grafik/CSV, lalu pindahkan notebook dan hasil penting ke GitHub.

**Proyek:** klasifikasi 6 kelas sampah (`cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`) menggunakan ResNet18. Notebook membandingkan tiga mode:

- `feature`: bobot pretrained dibekukan; classifier akhir dilatih.
- `partial`: bobot pretrained digunakan; `layer4` dan classifier akhir dilatih.
- `scratch`: model diinisialisasi tanpa bobot pretrained.

> **Penting:** hasil yang keluar di Colab akan menjadi hasil final praktikum. Angka dari eksperimen lokal sebelumnya tidak otomatis sama karena lingkungan, bobot, urutan batch, dan perangkat dapat berbeda. Jalankan semua sel dari atas ke bawah dan gunakan angka yang benar-benar muncul di notebook ini.

## Cara menjalankan
1. Buka `Runtime → Change runtime type → T4 GPU` (atau GPU yang tersedia).
2. Siapkan ZIP dataset dengan format folder per kelas, atau format `train/kelas`, `val/kelas`, dan `test/kelas`.
3. Jalankan sel secara berurutan. Simpan notebook (`File → Save a copy in Drive`) dan setelah selesai pindahkan ke repositori GitHub milik sendiri.

## 1. Konfigurasi eksperimen

In [ ]:
SEED = 42
EPOCHS = 10
BATCH_SIZE = 16
IMG_SIZE = 224
NUM_WORKERS = 0  # dibuat 0 agar lebih stabil di berbagai sesi Colab
MODES = ["feature", "partial", "scratch"]

DATA_DIR = "/content/dataset_extracted"
OUTPUT_DIR = "/content/outputs_resnet18"

print("Konfigurasi:")
print({"seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
       "img_size": IMG_SIZE, "modes": MODES})

## 2. Unggah dan ekstrak ZIP dataset

In [ ]:
import os, re, shutil, zipfile, random, time, copy, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from google.colab import files

uploaded = files.upload()
zip_candidates = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_candidates:
    raise ValueError("Unggah satu file ZIP yang berisi dataset gambar.")

zip_name = zip_candidates[0]
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(DATA_DIR)

print(f"Dataset diekstrak dari: {zip_name}")
print(f"Lokasi data: {DATA_DIR}")
print("Contoh isi folder:")
for p in list(Path(DATA_DIR).iterdir())[:15]:
    print(" -", p.name)

## 3. Baca struktur dataset dan buat pembagian train/val/test

In [ ]:
from sklearn.model_selection import train_test_split

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_ALIASES = {
    "train": "train", "training": "train",
    "val": "val", "valid": "val", "validation": "val",
    "test": "test", "testing": "test"
}

def clean_token(s):
    return re.sub(r"[^a-z0-9]+", "", s.lower())

def get_split_position(parts):
    for i, part in enumerate(parts[:-1]):
        key = clean_token(part)
        if key in SPLIT_ALIASES:
            return i, SPLIT_ALIASES[key]
    return None, None

image_paths = sorted(
    str(p) for p in Path(DATA_DIR).rglob("*")
    if p.is_file() and p.suffix.lower() in IMG_EXT and not any(x.startswith(".") for x in p.parts)
)
if not image_paths:
    raise ValueError("Tidak ada gambar yang ditemukan. Periksa isi ZIP dan format foldernya.")

rel_parts = [Path(p).relative_to(DATA_DIR).parts for p in image_paths]
split_info = [get_split_position(parts) for parts in rel_parts]
all_have_split = all(i is not None for i, _ in split_info)
splits_found = {s for _, s in split_info if s is not None}
rows = []

# Jika seluruh gambar berada pada folder train/val/test yang lengkap, pertahankan pembagian itu.
use_existing_split = all_have_split and splits_found == {"train", "val", "test"}
if use_existing_split:
    for path, parts, (split_idx, split) in zip(image_paths, rel_parts, split_info):
        kelas = parts[split_idx + 1] if split_idx + 1 < len(parts) - 1 else None
        if kelas:
            rows.append({"path": path, "kelas": kelas, "split": split})
else:
    # Dataset belum dibagi, atau split lengkap tidak ditemukan: deteksi folder kelas lalu split stratified 70/15/15.
    if all_have_split:
        # Bila struktur memiliki nama split tetapi tidak lengkap, ambil label kelas setelah folder split.
        for path, parts, (split_idx, split) in zip(image_paths, rel_parts, split_info):
            kelas = parts[split_idx + 1] if split_idx + 1 < len(parts) - 1 else None
            if kelas:
                rows.append({"path": path, "kelas": kelas})
    else:
        parent_parts = [parts[:-1] for parts in rel_parts]
        max_depth = min(len(p) for p in parent_parts)
        chosen_depth = None
        for depth in range(max_depth):
            labels = [p[depth] for p in parent_parts if len(p) > depth]
            counts = pd.Series(labels).value_counts()
            if 2 <= len(counts) <= 30 and counts.min() >= 2:
                chosen_depth = depth
                break
        if chosen_depth is None:
            # Fallback untuk ZIP sederhana: nama folder yang langsung memuat gambar dianggap kelas.
            classes_guess = [Path(p).parent.name for p in image_paths]
            if len(set(classes_guess)) < 2:
                raise ValueError("Folder kelas tidak terdeteksi. Susun gambar ke folder per kelas sebelum membuat ZIP.")
            labels = classes_guess
        else:
            labels = [p[chosen_depth] for p in parent_parts]
        rows = [{"path": path, "kelas": label} for path, label in zip(image_paths, labels)]

    df_all = pd.DataFrame(rows).drop_duplicates(subset=["path"]).reset_index(drop=True)
    if df_all.empty or df_all["kelas"].nunique() < 2:
        raise ValueError("Kelas dataset tidak terdeteksi dengan benar. Periksa susunan folder ZIP.")
    # Pisahkan 70% train, 15% validation, 15% test dengan distribusi kelas yang sebanding.
    try:
        train_df, temp_df = train_test_split(
            df_all, test_size=0.30, random_state=SEED, stratify=df_all["kelas"]
        )
        val_df, test_df = train_test_split(
            temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["kelas"]
        )
    except ValueError as e:
        raise ValueError("Sebagian kelas terlalu sedikit untuk split stratified. Tambahkan gambar per kelas.") from e
    train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()
    train_df["split"] = "train"; val_df["split"] = "val"; test_df["split"] = "test"
    df = pd.concat([train_df, val_df, test_df], ignore_index=True)

if use_existing_split:
    df = pd.DataFrame(rows).drop_duplicates(subset=["path"]).reset_index(drop=True)

# Normalisasi tipe teks dan urutkan agar urutan evaluasi mudah direproduksi.
df["kelas"] = df["kelas"].astype(str)
df["split"] = df["split"].astype(str)
df = df[df["split"].isin(["train", "val", "test"])].sort_values(["split", "kelas", "path"]).reset_index(drop=True)
required_splits = {"train", "val", "test"}
if set(df["split"].unique()) != required_splits:
    raise ValueError("Dataset harus memiliki gambar untuk train, val, dan test.")
if df["kelas"].nunique() < 2:
    raise ValueError("Dataset memerlukan minimal dua kelas.")

CLASSES = sorted(df["kelas"].unique())
class_to_idx = {name: i for i, name in enumerate(CLASSES)}
df["label"] = df["kelas"].map(class_to_idx).astype(int)

counts = pd.crosstab(df["split"], df["kelas"]).reindex(["train", "val", "test"]).fillna(0).astype(int)
counts["TOTAL"] = counts.sum(axis=1)
counts.loc["TOTAL"] = counts.sum(axis=0)
display(counts)
print(f"\nTotal gambar: {len(df)}")
print("Kelas:", CLASSES)
print("Split dipakai dari ZIP:", use_existing_split)

if (df.groupby("kelas").size() < 3).any():
    print("PERINGATAN: setidaknya ada kelas dengan kurang dari 3 gambar; hasil evaluasi mungkin tidak stabil.")

## 4. Visualisasi contoh gambar dan distribusi kelas

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts_by_class = df.groupby(["split", "kelas"]).size().unstack(fill_value=0)
counts_by_class.plot(kind="bar", ax=ax)
ax.set_title("Jumlah gambar per kelas dan split")
ax.set_xlabel("Split")
ax.set_ylabel("Jumlah gambar")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

n_show = min(12, len(df))
sample_df = df.sample(n=n_show, random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    try:
        with Image.open(row["path"]) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(f"{row['kelas']} | {row['split']}", fontsize=9)
    except Exception as exc:
        ax.set_title(f"Gagal: {Path(row['path']).name}", fontsize=8)
    ax.axis("off")
for ax in list(axes.flat)[n_show:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Siapkan dataset, transformasi, dan DataLoader

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("PERINGATAN: GPU tidak aktif. Pilih Runtime → Change runtime type → GPU untuk training lebih cepat.")

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class TrashImageDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row["path"]) as im:
            image = im.convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(row["label"])

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

generator = torch.Generator().manual_seed(SEED)
train_ds = TrashImageDataset(train_df, train_transform)
val_ds = TrashImageDataset(val_df, eval_transform)
test_ds = TrashImageDataset(test_df, eval_transform)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                      pin_memory=(device.type == "cuda"), generator=generator)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                    pin_memory=(device.type == "cuda"))
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                     pin_memory=(device.type == "cuda"))

print(f"Train: {len(train_ds)} | Validation: {len(val_ds)} | Test: {len(test_ds)}")
print("Jumlah kelas:", len(CLASSES))

## 6. Training ResNet18 pada mode feature, partial, dan scratch

In [ ]:
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)


def build_model(mode):
    if mode == "scratch":
        model = models.resnet18(weights=None)
    else:
        # Bobot pretrained ImageNet digunakan untuk feature dan partial.
        model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, len(CLASSES))

    if mode == "feature":
        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True
    elif mode == "partial":
        for param in model.parameters():
            param.requires_grad = False
        for param in model.layer4.parameters():
            param.requires_grad = True
        for param in model.fc.parameters():
            param.requires_grad = True
    elif mode != "scratch":
        raise ValueError(f"Mode tidak dikenal: {mode}")
    return model


def run_epoch(model, loader, criterion, optimizer=None, mode=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    # Pertahankan batch-normalization pada bagian yang dibekukan agar statistiknya tidak ikut berubah.
    if training and mode in ("feature", "partial"):
        for module in [model.conv1, model.bn1, model.layer1, model.layer2, model.layer3]:
            module.eval()
        if mode == "feature":
            model.layer4.eval()
        model.fc.train()

    total_loss = 0.0
    all_true, all_pred = [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            all_true.extend(y.detach().cpu().numpy().tolist())
            all_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())
    return total_loss / len(loader.dataset), accuracy_score(all_true, all_pred), np.array(all_true), np.array(all_pred)


histories = {}
trained_models = {}
summary_rows = []

for mode in MODES:
    print("\n" + "=" * 65)
    print(f"Mulai training mode: {mode}")
    model = build_model(mode).to(device)
    criterion = nn.CrossEntropyLoss()

    if mode == "partial":
        param_groups = [
            {"params": model.layer4.parameters(), "lr": 1e-4},
            {"params": model.fc.parameters(), "lr": 1e-3},
        ]
    else:
        param_groups = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_epoch = 0
    best_state = None
    start = time.time()

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc, _, _ = run_epoch(model, train_dl, criterion, optimizer, mode)
        val_loss, val_acc, _, _ = run_epoch(model, val_dl, criterion)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch:02d}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    elapsed = time.time() - start
    if best_state is not None:
        model.load_state_dict(best_state)
    weights_path = os.path.join(OUTPUT_DIR, "models", f"resnet18_{mode}_best.pth")
    torch.save(model.state_dict(), weights_path)
    histories[mode] = history
    trained_models[mode] = model
    summary_rows.append({
        "model": "resnet18", "mode": mode, "device": str(device),
        "best_epoch": best_epoch, "best_val_accuracy": best_val_acc,
        "training_seconds": elapsed, "epochs_requested": EPOCHS,
        "batch_size": BATCH_SIZE, "img_size": IMG_SIZE, "weights_path": weights_path,
    })
    print(f"Selesai: best_epoch={best_epoch}, best_val_accuracy={best_val_acc:.4f}, waktu={elapsed:.1f} detik")

print("\nSemua mode selesai dilatih.")

## 7. Grafik akurasi dan loss per epoch

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for mode, h in histories.items():
    ep = range(1, len(h["train_acc"]) + 1)
    ax.plot(ep, h["train_acc"], marker="o", label=f"{mode} train")
    ax.plot(ep, h["val_acc"], marker=".", linestyle="--", label=f"{mode} val")
ax.set_title("Akurasi training dan validasi per epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.02)
ax.legend(ncol=2)
ax.grid(True, alpha=0.25)
plt.tight_layout()
acc_plot_path = os.path.join(OUTPUT_DIR, "akurasi_per_epoch.png")
plt.savefig(acc_plot_path, dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for mode, h in histories.items():
    ep = range(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], marker="o", label=f"{mode} train")
    ax.plot(ep, h["val_loss"], marker=".", linestyle="--", label=f"{mode} val")
ax.set_title("Loss training dan validasi per epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend(ncol=2)
ax.grid(True, alpha=0.25)
plt.tight_layout()
loss_plot_path = os.path.join(OUTPUT_DIR, "loss_per_epoch.png")
plt.savefig(loss_plot_path, dpi=160, bbox_inches="tight")
plt.show()

## 8. Evaluasi data test dan perbandingan semua mode

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

history_rows = []
for mode, h in histories.items():
    for i in range(EPOCHS):
        history_rows.append({"mode": mode, "epoch": i + 1,
            "train_loss": h["train_loss"][i], "train_accuracy": h["train_acc"][i],
            "val_loss": h["val_loss"][i], "val_accuracy": h["val_acc"][i]})
history_df = pd.DataFrame(history_rows)
history_df.to_csv(os.path.join(OUTPUT_DIR, "riwayat_epoch.csv"), index=False)

metrics_rows = []
predictions = {}
for row in summary_rows:
    mode = row["mode"]
    model = trained_models[mode]
    test_loss, test_acc, y_true, y_pred = run_epoch(model, test_dl, nn.CrossEntropyLoss())
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    result = dict(row)
    result.update({"test_loss": test_loss, "test_accuracy": test_acc,
                   "test_precision_weighted": precision,
                   "test_recall_weighted": recall, "test_f1_weighted": f1,
                   "test_images": len(test_ds), "classes": ", ".join(CLASSES)})
    metrics_rows.append(result)
    predictions[mode] = (y_true, y_pred)

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(os.path.join(OUTPUT_DIR, "perbandingan_metrik.csv"), index=False)
display(metrics_df[["mode", "best_epoch", "best_val_accuracy", "test_loss", "test_accuracy",
                    "test_precision_weighted", "test_recall_weighted", "test_f1_weighted", "training_seconds"]
                  ].sort_values("best_val_accuracy", ascending=False).style.format({
                      "best_val_accuracy": "{:.2%}", "test_accuracy": "{:.2%}",
                      "test_precision_weighted": "{:.2%}", "test_recall_weighted": "{:.2%}",
                      "test_f1_weighted": "{:.2%}", "test_loss": "{:.4f}",
                      "training_seconds": "{:.1f}"
                  }))

print("Metrik tersimpan ke:", os.path.join(OUTPUT_DIR, "perbandingan_metrik.csv"))

## 9. Confusion matrix dan classification report

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Pilih konfigurasi berdasarkan akurasi validasi terbaik; data test tidak dipakai untuk memilih model.
best_mode = metrics_df.sort_values(["best_val_accuracy", "best_epoch"], ascending=[False, True]).iloc[0]["mode"]
print("Mode terpilih berdasarkan validation accuracy:", best_mode)
y_true, y_pred = predictions[best_mode]
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES))))
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(ax=ax, xticks_rotation=40, colorbar=False, values_format="d")
ax.set_title(f"Confusion Matrix - ResNet18 {best_mode} (test)")
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_test.png")
plt.savefig(cm_path, dpi=160, bbox_inches="tight")
plt.show()

report_dict = classification_report(y_true, y_pred, labels=list(range(len(CLASSES))),
                                    target_names=CLASSES, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(os.path.join(OUTPUT_DIR, "classification_report.csv"))
display(report_df.style.format("{:.3f}"))

## 10. Analisis gambar yang salah diprediksi

In [ ]:
y_true, y_pred = predictions[best_mode]
error_rows = []
for idx, (true_idx, pred_idx) in enumerate(zip(y_true, y_pred)):
    if true_idx != pred_idx:
        row = test_df.iloc[idx]
        error_rows.append({"path": row["path"], "nama_file": Path(row["path"]).name,
                           "kelas_asli": CLASSES[int(true_idx)],
                           "prediksi": CLASSES[int(pred_idx)]})
errors_df = pd.DataFrame(error_rows)
errors_path = os.path.join(OUTPUT_DIR, "analisis_kesalahan.csv")
errors_df.to_csv(errors_path, index=False)
print(f"Jumlah prediksi salah: {len(errors_df)} dari {len(test_df)} gambar test")
if len(errors_df):
    display(errors_df.head(20))
    n_show = min(8, len(errors_df))
    fig, axes = plt.subplots(2, 4, figsize=(12, 7))
    for ax, (_, row) in zip(axes.flat, errors_df.head(n_show).iterrows()):
        with Image.open(row["path"]) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(f"Asli: {row['kelas_asli']}\nPred: {row['prediksi']}", fontsize=9)
        ax.axis("off")
    for ax in list(axes.flat)[n_show:]: ax.axis("off")
    plt.tight_layout()
    error_plot_path = os.path.join(OUTPUT_DIR, "contoh_prediksi_salah.png")
    plt.savefig(error_plot_path, dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Tidak ada kesalahan prediksi pada test set untuk mode terpilih.")

## 11. Simpan hasil dan buat paket untuk GitHub

In [ ]:
# Simpan konfigurasi, hasil, dan informasi pembagian data.
config = {
    "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "img_size": IMG_SIZE, "device": str(device), "classes": CLASSES,
    "train_images": int(len(train_ds)), "val_images": int(len(val_ds)),
    "test_images": int(len(test_ds)), "best_mode_by_validation": best_mode
}
with open(os.path.join(OUTPUT_DIR, "config_eksperimen.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

# README hasil untuk disertakan ke repositori; tidak menyertakan dataset mentah agar repo tidak terlalu besar.
chosen = metrics_df.loc[metrics_df["mode"] == best_mode].iloc[0]
rows_md = [
    "# Transfer Learning ResNet18 — Klasifikasi Citra Sampah",
    "",
    "Notebook praktikum dijalankan di Google Colab. Notebook utama membandingkan mode `feature`, `partial`, dan `scratch` pada ResNet18.",
    "",
    "## Dataset",
    f"- Kelas: {', '.join(CLASSES)}",
    f"- Training: {len(train_ds)} gambar; validation: {len(val_ds)} gambar; test: {len(test_ds)} gambar.",
    f"- Input: {IMG_SIZE}×{IMG_SIZE}; batch size: {BATCH_SIZE}; epoch: {EPOCHS}; seed: {SEED}.",
    "",
    "## Ringkasan evaluasi",
    "| Mode | Best epoch | Best val accuracy | Test accuracy | Weighted F1 | Waktu (detik) |",
    "|---|---:|---:|---:|---:|---:|"
]
for _, r in metrics_df.iterrows():
    rows_md.append(f"| {r['mode']} | {int(r['best_epoch'])} | {r['best_val_accuracy']:.2%} | {r['test_accuracy']:.2%} | {r['test_f1_weighted']:.2%} | {r['training_seconds']:.1f} |")
rows_md += [
    "",
    f"Mode yang dipilih berdasarkan validation accuracy: **{best_mode}** (validation accuracy {chosen['best_val_accuracy']:.2%}). Pemilihan tidak menggunakan test accuracy.",
    "",
    "## Artefak",
    "- `perbandingan_metrik.csv`: metrik test dan waktu training untuk tiap mode.",
    "- `riwayat_epoch.csv`: loss dan akurasi train/validation per epoch.",
    "- `classification_report.csv`: precision, recall, F1 per kelas dan agregat.",
    "- `analisis_kesalahan.csv`: daftar sampel test yang salah diprediksi oleh mode terpilih.",
    "- `akurasi_per_epoch.png`, `loss_per_epoch.png`, `confusion_matrix_test.png`: grafik evaluasi.",
    "",
    "> Catatan: hasil ini berasal dari run yang terekam pada notebook Colab. Bila notebook dijalankan ulang, nilai metrik dan waktu dapat berubah.",
]
readme_path = os.path.join(OUTPUT_DIR, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write("\n".join(rows_md) + "\n")

# Buat paket publikasi berisi hasil non-biner. Model .pth sengaja tidak dimasukkan ke paket default
# agar repositori tidak menjadi terlalu besar; file model tetap tersedia di folder outputs_resnet18/models.
package_root = "/content/resnet18_github_package"
if os.path.exists(package_root): shutil.rmtree(package_root)
os.makedirs(os.path.join(package_root, "results"), exist_ok=True)
shutil.copy(readme_path, os.path.join(package_root, "README.md"))
for name in [
    "perbandingan_metrik.csv", "riwayat_epoch.csv", "classification_report.csv",
    "analisis_kesalahan.csv", "config_eksperimen.json", "akurasi_per_epoch.png",
    "loss_per_epoch.png", "confusion_matrix_test.png", "contoh_prediksi_salah.png"
]:
    src = os.path.join(OUTPUT_DIR, name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(package_root, "results", name))

zip_out = "/content/resnet18_github_package.zip"
if os.path.exists(zip_out): os.remove(zip_out)
shutil.make_archive(zip_out[:-4], "zip", package_root)

print("Paket publikasi dibuat:", zip_out)
print("Metrik:", os.path.join(OUTPUT_DIR, "perbandingan_metrik.csv"))
print("Model terbaik per mode ada di:", os.path.join(OUTPUT_DIR, "models"))
print("\nUnduh paket hasil ke komputer:")
files.download(zip_out)

## 12. Simpan notebook ke GitHub

**Jangan commit ke repositori contoh dosen/orang lain tanpa izin.** Buat repositori milik sendiri atau fork repo tersebut ke akun GitHub kamu.

### Opsi paling mudah: dari Google Colab
1. Pastikan notebook sudah tersimpan dan nama file-nya jelas, misalnya `ResNet18-Transfer-Learning-Sampah.ipynb`.
2. Di Colab pilih **File → Save a copy in GitHub** (jika opsi ini tersedia dan akun GitHub sudah terhubung).
3. Pilih repositori milik sendiri, branch `main`, dan tulis pesan commit, misalnya `Add ResNet18 transfer learning experiment`.
4. Di halaman repo GitHub, pastikan notebook tampil dan bisa dibuka. Klik badge/opsi **Open in Colab** bila ingin menambahkan tautan pada README.
5. Upload `resnet18_github_package.zip` atau ekstrak isinya lalu letakkan `README.md` dan folder `results/` ke repo. Bila mengunggah satu ZIP saja, pastikan instruksi praktikum memang menerima hasil dalam format ZIP.

### Screenshot dokumentasi yang harus diambil
- **Dokumentasi 1 — Setup Colab:** notebook terbuka dan `Runtime` menunjukkan GPU aktif (jangan hanya screenshot halaman kosong).
- **Dokumentasi 2 — Dataset:** tabel jumlah gambar train/val/test per kelas serta contoh gambar yang ditampilkan notebook.
- **Dokumentasi 3 — Training:** output epoch untuk `feature`, `partial`, dan `scratch`.
- **Dokumentasi 4 — Evaluasi:** tabel metrik test dan classification report.
- **Dokumentasi 5 — Grafik:** kurva accuracy/loss serta confusion matrix.
- **Dokumentasi 6 — GitHub:** halaman repo milik sendiri yang menampilkan notebook, README, dan folder hasil.

Gunakan screenshot asli dari hasil run kamu, bukan gambar contoh. Setelah run Colab selesai, angka dalam laporan Word harus disalin dari output notebook Colab ini.